# 06 — Check SoilGrids against Soil Health Card samples

SoilGrids is a global model, and its own 90% interval for Ludhiana's 0–30 cm
stock spans 2.2–27.7 MtC (`05_soilgrids_uncertainty.ipynb`). The only way to
narrow that is local measurement. India's Soil Health Card programme has
tested organic carbon at hundreds of thousands of geotagged farm points in
Punjab; this notebook compares those measurements with SoilGrids at the same
250 m cells.

**Input.** The Punjab Soil Health Card table from Dataful
(https://dataful.in/datasets/24083/, Ministry of Agriculture and Farmers
Welfare; paid download), saved anywhere under `IIRS/Carbon_Stocks/` as CSV, XLSX or
Parquet. The notebook finds it by its columns (`latitude`, `longitude`,
`organic_carbon_percentage`).

**Three things make the comparison fair, and each is stated, not hidden:**

1. **Method.** Soil-test labs measure organic carbon by Walkley–Black wet
   oxidation, which recovers only part of it; the conventional correction is
   × 1.32 (recovery ≈ 76%, Walkley & Black 1934; De Vos et al. 2007 report a
   wide range, so both raw and corrected values are shown). SoilGrids is
   trained mostly on total organic carbon.
2. **Depth.** Soil Health Card samples are plough-layer samples, usually
   0–15 cm. SoilGrids `soc_030` is a 0–30 cm thickness-weighted mean, which in
   cultivated soils should be *lower* than the 0–15 cm value. So a SoilGrids
   value above the corrected test value is a real overestimate, not a depth
   artefact.
3. **Support.** A sample is a point; a SoilGrids cell is 250 m. Individual
   agreement will be noisy; the district median ratio is the robust number.


In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))
import config as C

WB_FACTOR = 1.32            # Walkley-Black recovery correction (conventional)
OC_RANGE = (0.02, 3.0)      # % organic carbon accepted as a plausible lab value in Punjab's alluvium
NEEDED = {"latitude", "longitude", "organic_carbon_percentage"}

def read_any(p):
    if p.suffix == ".csv":
        return pd.read_csv(p, low_memory=False)
    if p.suffix in (".xlsx", ".xls"):
        return pd.read_excel(p)
    if p.suffix == ".parquet":
        return pd.read_parquet(p)

OUT = Path(os.environ.get("SHC_OUT", C.RESULTS))       # results folder (overridable for tests)
found = Path(os.environ["SHC_FILE"]) if "SHC_FILE" in os.environ else None
for p in ([] if found else sorted((C.RAW).rglob("*"))):
    if p.suffix.lower() not in (".csv", ".xlsx", ".xls", ".parquet") or p.stat().st_size > 2e9:
        continue
    try:
        head = read_any(p) if p.suffix != ".csv" else pd.read_csv(p, nrows=5)
    except Exception:
        continue
    cols = {c.strip().lower() for c in head.columns}
    if NEEDED <= cols:
        found = p
        break

if found is None:
    raise SystemExit("No Soil Health Card table in IIRS/Carbon_Stocks/ yet (needs latitude, longitude, "
                     "organic_carbon_percentage). Download it from https://dataful.in/datasets/24083/ and re-run.")
shc = read_any(found)
shc.columns = [c.strip().lower() for c in shc.columns]
print(f"{found.name}: {len(shc):,} rows, columns: {list(shc.columns)}")


## Clean and place each sample on the grid

In [ ]:
raw_n = len(shc)
if "district" in shc:
    shc = shc[shc["district"].astype(str).str.contains("ludhiana", case=False, na=False)]
shc = shc.assign(
    lat=pd.to_numeric(shc["latitude"], errors="coerce"),
    lon=pd.to_numeric(shc["longitude"], errors="coerce"),
    oc_pct=pd.to_numeric(shc["organic_carbon_percentage"], errors="coerce"),
)
steps = {"rows read": raw_n, "in Ludhiana district": len(shc)}
shc = shc.dropna(subset=["lat", "lon", "oc_pct"])
steps["with coordinates and a value"] = len(shc)
shc = shc[shc["oc_pct"].between(*OC_RANGE)]
steps[f"value within {OC_RANGE[0]}-{OC_RANGE[1]} %"] = len(shc)
dup_cols = [c for c in ["fiscal_year", "lat", "lon", "oc_pct"] if c in shc or c in ("lat", "lon", "oc_pct")]
shc = shc.drop_duplicates(subset=dup_cols)
steps["after removing exact duplicates"] = len(shc)

# Lattice position: cell corners sit at integer multiples of the step from (0, 0)
shc["grid_row"] = np.floor(shc["lat"] / C.GRID_STEP_DEG).astype(int)
shc["grid_col"] = np.floor(shc["lon"] / C.GRID_STEP_DEG).astype(int)
grid = pd.read_csv(C.DATA / "v2" / "ludhiana_cells_v2.csv.gz",
                   usecols=["cell_id", "grid_row", "grid_col", "in_district_frac", "soil_valid_frac",
                            "soc_030", "bdod_030", "cfvo_030", "ocs_030_t_ha", "cropland_frac"])
m = shc.merge(grid, on=["grid_row", "grid_col"], how="inner")
m = m[(m["in_district_frac"] > 0) & (m["soil_valid_frac"] > 0) & m["soc_030"].notna()]
steps["inside the district grid with SoilGrids soil"] = len(m)
for k, v in steps.items():
    print(f"{v:>9,}  {k}")
if len(m) < 30:
    raise SystemExit("Too few matched samples for a meaningful comparison.")


## Compare at the same cells

In [ ]:
m["shc_gkg_raw"] = m["oc_pct"] * 10                    # % -> g/kg
m["shc_gkg_corr"] = m["shc_gkg_raw"] * WB_FACTOR
m["ratio_raw"] = m["soc_030"] / m["shc_gkg_raw"]
m["ratio_corr"] = m["soc_030"] / m["shc_gkg_corr"]

def summary_block(d):
    return {
        "samples": int(len(d)), "cells": int(d["cell_id"].nunique()),
        "shc_oc_g_kg_median_raw": round(float(d["shc_gkg_raw"].median()), 2),
        "shc_oc_g_kg_median_corrected": round(float(d["shc_gkg_corr"].median()), 2),
        "soilgrids_soc_030_g_kg_median": round(float(d["soc_030"].median()), 2),
        "median_ratio_soilgrids_over_shc_raw": round(float(d["ratio_raw"].median()), 3),
        "median_ratio_soilgrids_over_shc_corrected": round(float(d["ratio_corr"].median()), 3),
        "spearman_r": round(float(d["soc_030"].corr(d["shc_gkg_raw"], method="spearman")), 3),
    }

out = {"source": found.name, "walkley_black_factor": WB_FACTOR, "cleaning": steps,
       "all": summary_block(m)}
if "fiscal_year" in m:
    out["by_year"] = {str(y): summary_block(g) for y, g in m.groupby("fiscal_year") if len(g) >= 30}
print(json.dumps(out, indent=2))


### What it implies for the district stock

If SoilGrids' carbon concentration is too high by the median ratio, its stock
is too high by roughly the same factor (bulk density and coarse fragments are
unchanged). Scaling the census headline by 1 / ratio gives an indicative,
soil-test-anchored stock. Because the tests are 0–15 cm and carbon declines
with depth, this is still an upper bound.

In [ ]:
summary = json.loads((C.RESULTS / "district_summary.json").read_text())
headline = summary["soil"]["stock_mtc"]["estimate"]
r_raw, r_corr = out["all"]["median_ratio_soilgrids_over_shc_raw"], out["all"]["median_ratio_soilgrids_over_shc_corrected"]
out["implied_stock_mtc"] = {
    "soilgrids_headline": headline,
    "scaled_to_shc_corrected": round(headline / r_corr, 2),
    "scaled_to_shc_raw": round(headline / r_raw, 2),
    "note": "0-15 cm test values applied to 0-30 cm; an upper bound if carbon declines with depth",
}
print(json.dumps(out["implied_stock_mtc"], indent=2))
(OUT / "shc_validation.json").write_text(json.dumps(out, indent=2))
print("Wrote IIRS/Carbon_Stocks/results/shc_validation.json")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
lim = float(np.nanpercentile(np.r_[m["soc_030"], m["shc_gkg_corr"]], 99.5))
ax[0].hexbin(m["shc_gkg_corr"], m["soc_030"], gridsize=45, extent=(0, lim, 0, lim), mincnt=1, cmap="YlOrBr")
ax[0].plot([0, lim], [0, lim], color="#2A231C", lw=1)
ax[0].set(xlabel="Soil Health Card OC, g/kg (x 1.32 Walkley-Black)", ylabel="SoilGrids SOC 0-30 cm, g/kg",
          title="Same 250 m cell; line = 1:1")
ax[1].hist(np.log2(m["ratio_corr"].clip(1 / 16, 16)), bins=60, color="#8F4E1C")
ax[1].axvline(0, color="#2A231C", lw=1)
ax[1].set(xlabel="log2(SoilGrids / soil test); 0 = agreement, 1 = SoilGrids twice as high", ylabel="samples",
          title=f"Median ratio {r_corr:.2f} (corrected), {r_raw:.2f} (raw)")
plt.tight_layout(); plt.savefig(OUT / "fig_shc_validation.png", dpi=130); plt.show()


## Soil-test-calibrated map: regression kriging

The comparison above only rescales SoilGrids. With soil tests at many cells,
organic carbon can be mapped from the tests themselves (Hengl et al., 2004):

1. **Trend.** A quantile random forest predicts the cell-median soil-test value
   from the same covariates as the pipeline's models (`02`), plus SoilGrids SOC
   as one more covariate.
2. **Residual kriging.** The forest's residuals are interpolated by a Gaussian
   process on coordinates (an exponential covariance with a nugget), which is
   kriging with a fitted variogram.
3. **Validation.** Both steps are refitted inside 5-fold spatial-block
   cross-validation and compared with SoilGrids alone (rescaled by the median
   ratio).

The calibrated map gives a soil-test-anchored 0–30 cm stock,
SOC × BD × 30 cm × (1 − coarse fraction) with SoilGrids' bulk density and coarse
fragments. Because the tests are 0–15 cm, it is an upper bound.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import GroupKFold
from quantile_forest import RandomForestQuantileRegressor

feat = pd.read_parquet(C.RESULTS / "model_features.parquet")              # written by 02_carbon_pipeline
COV = [c for c in feat.columns if c not in ("cell_id", "grid_row", "grid_col", "ocs_030_t_ha", "bdod_030", "cfvo_030")]
pts = (m.groupby("cell_id").agg(shc=("shc_gkg_corr", "median"), n_samples=("shc_gkg_corr", "size")).reset_index()
       .merge(feat, on="cell_id", how="inner"))
xy_km = lambda d: np.c_[(d["lon"] - 75.85) * 95.5, (d["lat"] - 30.85) * 111.0]   # local km
print(f"{len(pts):,} cells with soil tests")

def make_gp():
    k = ConstantKernel(1.0) * Matern(length_scale=5.0, nu=0.5, length_scale_bounds=(0.2, 100)) + WhiteKernel(1.0)
    return GaussianProcessRegressor(kernel=k, normalize_y=True, random_state=C.RANDOM_STATE)

def fit_rk(d, max_gp=2500):
    rf = RandomForestQuantileRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.33, n_jobs=-1,
                                       random_state=C.RANDOM_STATE).fit(d[COV].to_numpy(float), d["shc"].to_numpy(float))
    oob = rf.predict(d[COV].to_numpy(float), quantiles="mean", oob_score=True)   # out-of-bag trend
    gsub = d.assign(res=d["shc"].to_numpy() - oob).sample(min(max_gp, len(d)), random_state=C.RANDOM_STATE)
    gp = make_gp().fit(xy_km(gsub), gsub["res"].to_numpy())
    return rf, gp

def predict_rk(rf, gp, d, batch=4000):
    trend = rf.predict(d[COV].to_numpy(float), quantiles="mean")
    xy = xy_km(d)
    resid = np.concatenate([gp.predict(xy[i:i + batch]) for i in range(0, len(xy), batch)])  # batches bound memory
    return trend, trend + resid

ratio = out["all"]["median_ratio_soilgrids_over_shc_corrected"]
y = pts["shc"].to_numpy(float)
p_sg, p_rf, p_rk = pts["soc_030"].to_numpy(float) / ratio, np.empty_like(y), np.empty_like(y)
for tr, te in GroupKFold(C.N_FOLDS).split(pts, y, C.spatial_blocks(pts["lon"], pts["lat"])):
    rf, gp = fit_rk(pts.iloc[tr])
    p_rf[te], p_rk[te] = predict_rk(rf, gp, pts.iloc[te])
sc = lambda p: {"r2": round(r2_score(y, p), 3), "rmse_g_kg": round(mean_squared_error(y, p) ** 0.5, 3)}
out["regression_kriging"] = {
    "cells": int(len(pts)), "samples": int(pts["n_samples"].sum()), "validation": "5-fold spatial-block CV (8x8)",
    "soilgrids_rescaled": sc(p_sg), "random_forest": sc(p_rf), "regression_kriging": sc(p_rk)}

rf, gp = fit_rk(pts)
inside = feat[feat["soc_030"].notna()].copy()
_, inside["soc_rk_g_kg"] = predict_rk(rf, gp, inside)
grid_area = pd.read_csv(C.DATA / "v2" / "ludhiana_cells_v2.csv.gz",
                        usecols=["cell_id", "cell_area_ha", "in_district_frac", "soil_valid_frac"])
inside = inside.merge(grid_area, on="cell_id")
stock = (C.soc_stock_tc_ha(inside["soc_rk_g_kg"].clip(lower=0), inside["bdod_030"], 30, inside["cfvo_030"] / 100)
         * inside["soil_valid_frac"] * inside["cell_area_ha"] * inside["in_district_frac"])
out["regression_kriging"]["stock_mtc_upper_bound"] = round(float(stock.sum()) / 1e6, 2)
out["regression_kriging"]["gp_kernel"] = str(gp.kernel_)
print(json.dumps(out["regression_kriging"], indent=2))
(OUT / "shc_validation.json").write_text(json.dumps(out, indent=2))